# Fake Job Posting Detection: Fraud Detection with Machine Learning and Deep Learning

**Dataset:** Kaggle "Real or Fake: Fake Job Posting Prediction" (`shivamb/real-or-fake-fake-jobposting-prediction`) | **Tools:** Python, scikit-learn, LightGBM, Keras | **Platform:** Google Colab

**Problem statement.** Job portals receive thousands of postings a day and a small share are fake. Scammers use them to steal personal data or collect "registration fees". Reading every posting by hand is not possible, so the portal needs a model that gives every posting a **fraud risk score** and sends the riskiest ones to human review first.

**Business objective.** Catch as many fake postings as possible without flooding the review team with false alarms.

**ML objective.** Predict the target `fraudulent` (1 = fake, 0 = real) from the posting text, the company details and the posting flags, and compare a simple baseline, a Random Forest and a tuned LightGBM model.

**DL objective.** Train a small Keras neural network on the same data and check honestly whether it beats the machine learning models.

**Dataset (expected columns).** `title`, `location`, `department`, `salary_range`, `company_profile`, `description`, `requirements`, `benefits` (text), `telecommuting`, `has_company_logo`, `has_questions` (0/1 flags), `employment_type`, `required_experience`, `required_education`, `industry`, `function` (categories), `job_id` (ID) and the target `fraudulent`. Section 1 checks these against the real file. **No number in this notebook is typed in by hand: every count, rate and metric is calculated when you run the cells.**

**How to run.** Runtime > Run all. A full run takes roughly 15 to 30 minutes on a free Colab CPU. Set `FAST_MODE = True` in the next cell for a quick test.

In [ ]:
!pip install -q imbalanced-learn shap kagglehub

import os, re, glob, time, math, random, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_validate, cross_val_predict
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.utils.class_weight import compute_class_weight
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, classification_report, roc_curve, precision_recall_curve)
from sklearn.inspection import permutation_importance
from scipy.stats import loguniform, randint, uniform
from imblearn.pipeline import Pipeline as ImbPipeline      # lets resampling run only on training folds
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from lightgbm import LGBMClassifier
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

warnings.filterwarnings("ignore")
SEED = 42
FAST_MODE = False                       # True = quick test run
N_TREES, N_ITER, EPOCHS = (60, 2, 3) if FAST_MODE else (300, 8, 80)
COST_FN, COST_FP = 10, 1                # ASSUMED costs: a missed scam is 10x worse than a false alarm (edit to your case)
random.seed(SEED); np.random.seed(SEED); keras.utils.set_random_seed(SEED)
sns.set_theme(style="whitegrid"); plt.rcParams.update({"figure.dpi": 100, "axes.titleweight": "bold"})
pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 60)
BLUE, RED = "#4C72B0", "#C44E52"; PALETTE = {0: BLUE, 1: RED}

def interpret(*lines):
    """Short data-driven interpretation shown under an output."""
    display(Markdown("**Interpretation**\n\n" + "\n".join(f"- {l}" for l in lines)))

---
# Phase 1: Data Understanding, Cleaning and EDA

We first load the data and check what is really in the file, because nothing is assumed. We then clean it, and explore it with charts to see what fake postings look like.

In [ ]:
def find_dataset_csv():
    """Download with kagglehub; if Kaggle cannot be reached, ask for a manual upload of fake_job_postings.csv."""
    csvs = []
    try:
        import kagglehub
        folder = kagglehub.dataset_download("shivamb/real-or-fake-fake-jobposting-prediction")
        csvs = glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True)
    except Exception as err:
        print("kagglehub failed:", repr(err))
    if not csvs:
        from google.colab import files
        csvs = list(files.upload().keys())
    return sorted(csvs, key=os.path.getsize, reverse=True)[0]

raw_df = pd.read_csv(find_dataset_csv())
df = raw_df.copy()
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
display(df.head(3)); df.info()

# Decide the role of every column from the real file
TARGET = "fraudulent"
assert TARGET in df.columns, f"Target '{TARGET}' not found. Columns: {list(df.columns)}"
ID_COLS   = [c for c in df.columns if c.lower() in ("job_id", "id")]
TEXT_COLS = [c for c in ["title", "company_profile", "description", "requirements", "benefits"] if c in df.columns]
SEMI_COLS = [c for c in ["location", "department", "salary_range"] if c in df.columns]
CAT_RAW   = [c for c in ["employment_type", "required_experience", "required_education", "industry", "function"] if c in df.columns]
BINARY_RAW = [c for c in df.columns if c != TARGET and pd.api.types.is_numeric_dtype(df[c]) and set(df[c].dropna().unique()) <= {0, 1}]
date_like = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]
interpret(f"Target: **{TARGET}**. ID columns: {ID_COLS}. Free text: {TEXT_COLS}. Binary flags: {BINARY_RAW}. Categories: {CAT_RAW}.",
          f"Date/time columns: {date_like or 'none'}" + (" so a time-based split is possible." if date_like else ", so a time-based split is not possible and we use a stratified split."))

In [ ]:
# ---- Data quality checks ----
missing = pd.DataFrame({"missing": df.isna().sum(), "missing_pct": (df.isna().mean() * 100).round(2)}).sort_values("missing_pct", ascending=False)
display(missing.head(10))

content_cols = [c for c in df.columns if c not in ID_COLS]
n_dup = int(df.duplicated(subset=content_cols).sum())
key = pd.util.hash_pandas_object(df[[c for c in content_cols if c != TARGET]].astype(str), index=False)
n_conflict = int((df.groupby(key)[TARGET].nunique() > 1).sum())
n_const = [c for c in df.columns if df[c].nunique() <= 1]
high_card = [c for c in SEMI_COLS + CAT_RAW if df[c].nunique() > 100]
q1, q3 = df["description"].fillna("").str.len().quantile([.25, .75]) if "description" in df.columns else (0, 0)
out_pct = (df["description"].fillna("").str.len() > q3 + 1.5 * (q3 - q1)).mean() * 100 if "description" in df.columns else 0

# ---- Cleaning ----
df_model = df.drop(columns=ID_COLS)                                   # IDs have no meaning and only allow memorising
for c in df_model.select_dtypes("object"):                            # blank strings become real missing values
    df_model[c] = df_model[c].apply(lambda v: (np.nan if v.strip() == "" else v.strip()) if isinstance(v, str) else v)
df_model = df_model.dropna(subset=[TARGET]); df_model[TARGET] = df_model[TARGET].astype(int)
df_model = df_model.drop_duplicates().reset_index(drop=True)          # duplicates in both train and test would leak

interpret(f"Most missing column: **{missing.index[0]}** ({missing['missing_pct'].iloc[0]}%). We keep these rows and later turn missingness into features, because an empty field can itself signal fraud.",
          f"{n_dup:,} rows repeat the content of another row, and {n_conflict} repeated-content groups carry conflicting labels. Duplicates were removed so one posting cannot sit in both train and test (data leakage).",
          f"Constant columns: {n_const or 'none'}. High-cardinality columns: {high_card or 'none'} (not one-hot encoded directly). Long descriptions: {out_pct:.1f}% are IQR outliers; we keep them and use a log transform plus RobustScaler.",
          f"Clean table: **{df_model.shape[0]:,} rows** (from {len(df):,}), fake rate {df[TARGET].mean() * 100:.2f}% before and {df_model[TARGET].mean() * 100:.2f}% after cleaning.")

### 1.1 Class imbalance
Fraud is rare. If only a small share of postings are fake, a model that always says "real" looks very accurate and catches nothing. We measure the imbalance first.

In [ ]:
counts = df_model[TARGET].value_counts().sort_index()
n_pos_all, n_neg_all = int(counts.get(1, 0)), int(counts.get(0, 0))
fraud_rate = df_model[TARGET].mean(); imbalance_ratio = n_neg_all / max(n_pos_all, 1)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(x=["Real (0)", "Fake (1)"], y=counts.values, palette=[BLUE, RED], ax=ax[0])
for i, v in enumerate(counts.values): ax[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontweight="bold")
ax[0].set_title("Postings by class")
ax[1].pie(counts.values, labels=["Real", "Fake"], colors=[BLUE, RED], autopct="%1.2f%%", startangle=90, wedgeprops=dict(width=0.45, edgecolor="white"))
ax[1].set_title("Class share")
plt.tight_layout(); plt.show()
interpret(f"{n_neg_all:,} real vs {n_pos_all:,} fake postings: fake postings are **{fraud_rate * 100:.2f}%** of the data (about {imbalance_ratio:.1f} real for every fake).",
          f"Always predicting 'real' would be {100 - fraud_rate * 100:.2f}% accurate and catch **zero** fraud, so we judge models with **recall, precision, F1, ROC-AUC and PR-AUC**, not accuracy.")

### 1.2 Missing fields and flags as fraud signals
Genuine employers usually fill in the company profile, requirements and logo. Thin postings may be riskier. We compare the fraud rate when a field is **missing** with when it is **present**, and the fraud rate for each 0/1 flag.

In [ ]:
rows = []
for c in df_model.columns:
    if c == TARGET: continue
    m = df_model[c].isna()
    if 0 < m.sum() < len(m): rows.append((c, m.sum(), df_model.loc[m, TARGET].mean() * 100, df_model.loc[~m, TARGET].mean() * 100))
gap = pd.DataFrame(rows, columns=["column", "n_missing", "fraud_if_missing", "fraud_if_present"])
gap["gap"] = gap["fraud_if_missing"] - gap["fraud_if_present"]; gap = gap.sort_values("gap", ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
g = gap.melt(id_vars="column", value_vars=["fraud_if_missing", "fraud_if_present"], var_name="state", value_name="fraud_rate")
sns.barplot(data=g, y="column", x="fraud_rate", hue="state", palette=[RED, BLUE], ax=ax[0]); ax[0].axvline(fraud_rate * 100, ls="--", c="grey")
ax[0].set_title("Fraud rate (%) when a field is missing vs present"); ax[0].set_ylabel("")
fl = pd.DataFrame([(f"{c} = {int(v)}", r * 100) for c in BINARY_RAW for v, r in df_model.groupby(c)[TARGET].mean().items()], columns=["label", "rate"])
if len(fl):
    sns.barplot(data=fl, y="label", x="rate", color=RED, ax=ax[1]); ax[1].axvline(fraud_rate * 100, ls="--", c="grey")
    ax[1].set_title("Fraud rate (%) by flag value"); ax[1].set_ylabel("")
plt.tight_layout(); plt.show()
top = gap.iloc[0]
interpret(f"Biggest gap: **{top['column']}** has fraud rate {top['fraud_if_missing']:.2f}% when missing vs {top['fraud_if_present']:.2f}% when present (dashed line = overall {fraud_rate * 100:.2f}%).",
          "Where the red 'missing' bar is far above the blue one, a missing-value flag is a useful feature, so we create these flags in feature engineering.")

### 1.3 Categories and text length
We check which categories and countries have more fraud (only groups with at least 30 postings, since tiny groups give unreliable rates), and whether fake postings are shorter or longer than real ones.

In [ ]:
df_model["country_eda"] = df_model["location"].fillna("Missing").str.split(",").str[0].str.strip() if "location" in df_model.columns else "Missing"
cat_show = [c for c in ["employment_type", "required_experience", "industry", "country_eda"] if c in df_model.columns]
fig, axes = plt.subplots(2, 3, figsize=(18, 9)); axes = axes.ravel()
best_cat = []
for ax, c in zip(axes, cat_show):
    t = df_model.groupby(c, dropna=False)[TARGET].agg(count="size", rate="mean").reset_index(); t[c] = t[c].fillna("Missing")
    t = t[t["count"] >= 30].sort_values("count", ascending=False).head(10).sort_values("rate", ascending=False); t["rate"] *= 100
    sns.barplot(data=t, y=c, x="rate", color=RED, ax=ax); ax.axvline(fraud_rate * 100, ls="--", c="grey"); ax.set_title(f"Fraud rate (%) by {c}"); ax.set_ylabel("")
    if len(t): best_cat.append(f"`{c}`: **{t.iloc[0][c]}** at {t.iloc[0]['rate']:.1f}% (n={int(t.iloc[0]['count'])})")
length = df_model[TEXT_COLS].fillna("").apply(lambda s: np.log1p(s.str.len())); length[TARGET] = df_model[TARGET].values
lm = length.melt(id_vars=TARGET, var_name="field", value_name="log_len")
sns.boxplot(data=lm, x="field", y="log_len", hue=TARGET, palette=PALETTE, ax=axes[4]); axes[4].set_title("Text length (log) by class"); axes[4].tick_params(axis="x", rotation=30)
for ax in axes[5:]: ax.axis("off")
plt.tight_layout(); plt.show()
med = df_model[TEXT_COLS].fillna("").apply(lambda s: s.str.len()).groupby(df_model[TARGET]).median().T
interpret("Highest fraud rate among frequent groups: " + "; ".join(best_cat) + ".",
          "Median text length (characters), real vs fake: " + "; ".join(f"{c} {r[0]:.0f} vs {r[1]:.0f}" for c, r in med.iterrows()) + ".",
          "Location is one signal among many. A high rate for a country may reflect where scammers target and must never be used to treat genuine employers there unfairly.")

### 1.4 Correlation and typical words
Correlation shows linear links only, so a weak value does not make a feature useless. For text we count the words that appear more in fake than in real postings (log-odds). This is for understanding only; the model learns its own weights from the training data.

In [ ]:
cdf = df_model[BINARY_RAW].copy()
for c in TEXT_COLS: cdf[f"{c}_len"] = df_model[c].fillna("").str.len(); cdf[f"{c}_missing"] = df_model[c].isna().astype(int)
cdf["salary_given"] = df_model["salary_range"].notna().astype(int) if "salary_range" in df_model.columns else 0
cdf[TARGET] = df_model[TARGET]; corr = cdf.corr()
tcorr = corr[TARGET].drop(TARGET).sort_values(key=np.abs, ascending=False)

txt = df_model[TEXT_COLS].fillna("").agg(" ".join, axis=1).str.lower().str.replace(r"#(url|email|phone)_[a-z0-9]+#", r" \1token ", regex=True)
cv = CountVectorizer(stop_words="english", min_df=20, binary=True, max_features=20000, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b")
W = cv.fit_transform(txt); words = np.array(cv.get_feature_names_out()); yy = df_model[TARGET].values
lo = np.log((np.asarray(W[yy == 1].sum(0)).ravel() + 1) / (yy == 1).sum()) - np.log((np.asarray(W[yy == 0].sum(0)).ravel() + 1) / (yy == 0).sum())
wt = pd.DataFrame({"word": words, "lo": lo}).sort_values("lo")

fig, ax = plt.subplots(1, 3, figsize=(20, 6.5))
sns.heatmap(corr, cmap="RdBu_r", center=0, annot=True, fmt=".2f", annot_kws={"size": 6}, cbar=False, ax=ax[0]); ax[0].set_title("Correlation matrix")
sns.barplot(data=wt.tail(12), y="word", x="lo", color=RED, ax=ax[1]); ax[1].set_title("Words typical of FAKE postings"); ax[1].set_ylabel("")
sns.barplot(data=wt.head(12), y="word", x="lo", color=BLUE, ax=ax[2]); ax[2].set_title("Words typical of REAL postings"); ax[2].set_ylabel("")
plt.tight_layout(); plt.show()
interpret(f"Strongest linear link with the target: **{tcorr.index[0]}** ({tcorr.iloc[0]:.3f}).",
          f"Words typical of fake postings: {', '.join(wt['word'].tail(8)[::-1])}. Typical of real postings: {', '.join(wt['word'].head(8))}.",
          "If these two word lists look clearly different, the text carries strong signals, so the model needs a proper text representation (TF-IDF).")

---
# Phase 2: Feature Engineering, Preprocessing and Split

**Leakage rule.** Every engineered feature is calculated **row by row from that posting's own content**, with no use of the target and no statistics from other rows. Anything that has to *learn* from data (TF-IDF vocabulary, scaling, one-hot categories) sits inside a pipeline fitted on the **training data only**.

| Feature | Why it can help |
|---|---|
| `<field>_missing`, `n_missing_fields` | Thin postings with many empty fields look suspicious |
| `log_<field>_len`, `log_total_words` | Scam text is often unusually short or long (log reduces skew) |
| `caps_ratio`, `exclam_count` | Shouting and many exclamation marks are common in scams |
| `url_count`, `email_count`, `phone_count` | Scammers push contact outside the platform (the data masks contacts as tokens) |
| `scam_phrase_count` | Count of typical scam wording (a simple expert rule, no labels used) |
| `no_logo_no_profile` | A combination of two weak signals |
| `country` | First part of `location`; easier to use than the full string |
| `text_all` | Cleaned join of all text fields, turned into TF-IDF numbers |

In [ ]:
MASK = re.compile(r"#(URL|EMAIL|PHONE)_[a-z0-9]+#", re.I); HTML = re.compile(r"<[^>]+>")
SCAM = ["work from home", "no experience", "earn", "weekly pay", "guaranteed", "immediately", "urgent",
        "wire transfer", "western union", "bank account", "registration fee", "investment", "unlimited"]

def build_features(frame):
    """Row-by-row features only, so it is safe to run before the split."""
    o = frame.copy()
    flag_src = [c for c in TEXT_COLS + SEMI_COLS + CAT_RAW if c in o.columns]
    for c in flag_src: o[f"{c}_missing"] = o[c].isna().astype(int)          # flags BEFORE filling
    o["n_missing_fields"] = o[[f"{c}_missing" for c in flag_src]].sum(axis=1)
    for c in TEXT_COLS: o[c] = o[c].fillna("")
    for c in CAT_RAW: o[c] = o[c].fillna("Missing")
    o["country"] = o["location"].fillna("").str.split(",").str[0].str.strip().replace("", "Missing") if "location" in o.columns else "Missing"
    for c in TEXT_COLS: o[f"log_{c}_len"] = np.log1p(o[c].str.len())
    raw = o[TEXT_COLS].agg(" ".join, axis=1)
    clean = lambda s: re.sub(r"\s+", " ", MASK.sub(lambda m: f" {m.group(1).lower()}token ", HTML.sub(" ", s)).lower()).strip()
    o["text_all"] = raw.apply(clean)
    o["log_total_words"] = np.log1p(o["text_all"].str.split().str.len())
    o["caps_ratio"] = raw.str.count(r"[A-Z]") / raw.str.count(r"[A-Za-z]").replace(0, 1)
    o["exclam_count"] = raw.str.count("!")
    for t in ["url", "email", "phone"]: o[f"{t}_count"] = raw.str.count(rf"#{t.upper()}_[a-z0-9]+#")
    o["scam_phrase_count"] = sum(o["text_all"].str.count(re.escape(p)) for p in SCAM)
    logo = o["has_company_logo"] if "has_company_logo" in o.columns else 1
    o["no_logo_no_profile"] = ((logo == 0) & (o["company_profile_missing"] == 1)).astype(int) if "company_profile_missing" in o.columns else 0
    return o

df_fe = build_features(df_model)
TEXT_FEATURE = "text_all"
CATEGORICAL_FEATURES = [c for c in CAT_RAW + ["country"] if c in df_fe.columns]
BINARY_FEATURES = BINARY_RAW + [c for c in df_fe.columns if c.endswith("_missing")] + ["no_logo_no_profile"]
NUMERIC_FEATURES = [f"log_{c}_len" for c in TEXT_COLS] + ["log_total_words", "caps_ratio", "exclam_count", "url_count", "email_count",
                                                          "phone_count", "scam_phrase_count", "n_missing_fields"]
const = [c for c in BINARY_FEATURES + NUMERIC_FEATURES if df_fe[c].nunique() <= 1]        # features that never change help nobody
BINARY_FEATURES = [c for c in BINARY_FEATURES if c not in const]; NUMERIC_FEATURES = [c for c in NUMERIC_FEATURES if c not in const]
ALL_FEATURES = [TEXT_FEATURE] + NUMERIC_FEATURES + CATEGORICAL_FEATURES + BINARY_FEATURES
display(df_fe[["title", "country", "n_missing_fields", "caps_ratio", "scam_phrase_count", "no_logo_no_profile", TARGET]].head())
interpret(f"{len(ALL_FEATURES)} model input columns: 1 text, {len(NUMERIC_FEATURES)} numeric, {len(CATEGORICAL_FEATURES)} categorical, {len(BINARY_FEATURES)} binary. Constant features dropped: {const or 'none'}.",
          "Left out on purpose: the ID, and the raw `location`, `department` and `salary_range` strings (high cardinality or messy; their information is kept through `country` and the missing flags).")

In [ ]:
def make_preprocessor(max_tfidf=3000):
    """Fresh, unfitted preprocessing: TF-IDF text, RobustScaler numbers, one-hot categories, 0/1 flags passed through."""
    return ColumnTransformer([
        ("text", TfidfVectorizer(max_features=max_tfidf, ngram_range=(1, 2), min_df=5, max_df=0.9, stop_words="english", sublinear_tf=True, dtype=np.float32), TEXT_FEATURE),
        ("num", RobustScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20, sparse_output=True), CATEGORICAL_FEATURES),
        ("bin", "passthrough", BINARY_FEATURES)], sparse_threshold=1.0)

# No date column exists, so we use a stratified 80/20 split (same fake rate in both parts). The test set is used once, at the end.
X, y = df_fe[ALL_FEATURES], df_fe[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
n_pos, n_neg = int(y_train.sum()), int((y_train == 0).sum())
display(pd.DataFrame({"Rows": [len(X_train), len(X_test)], "Fake": [n_pos, int(y_test.sum())], "Real": [n_neg, int((y_test == 0).sum())],
                      "Fake rate %": [y_train.mean() * 100, y_test.mean() * 100]}, index=["Train", "Test"]).round(3))

dummy_pred = DummyClassifier(strategy="most_frequent").fit(X_train, y_train).predict(X_test)
BASELINE_ACC = accuracy_score(y_test, dummy_pred)
shape = make_preprocessor().fit(X_train).transform(X_train).shape
interpret(f"Train: {len(X_train):,} postings ({n_pos} fake). Test: {len(X_test):,} postings ({int(y_test.sum())} fake). With so few fake test postings, one extra miss moves recall by about {100 / max(int(y_test.sum()), 1):.2f} points, so read small differences with care.",
          f"The 'always real' baseline reaches {BASELINE_ACC * 100:.2f}% accuracy with recall 0. After preprocessing (fitted on train only) the matrix has {shape[1]:,} columns.")

---
# Phase 3: Handling Class Imbalance

We compare four ways to deal with the imbalance using the same Logistic Regression and **5-fold stratified cross-validation on the training set**: no handling, `class_weight="balanced"`, random undersampling and **SMOTE** (synthetic fake postings). Resampling is placed **inside an imbalanced-learn pipeline**, so it touches only the training part of each fold, never the validation fold and never the test set. To make SMOTE meaningful we first compress the text with SVD, because SMOTE on thousands of sparse columns is not sensible.

In [ ]:
front = lambda: [("prep", make_preprocessor()), ("svd", TruncatedSVD(n_components=100, random_state=SEED))]
LR = lambda **k: LogisticRegression(max_iter=2000, random_state=SEED, **k)
strategies = {"No handling": ImbPipeline(front() + [("clf", LR())]),
              "class_weight": ImbPipeline(front() + [("clf", LR(class_weight="balanced"))]),
              "Undersampling": ImbPipeline(front() + [("s", RandomUnderSampler(random_state=SEED)), ("clf", LR())]),
              "SMOTE": ImbPipeline(front() + [("s", SMOTE(random_state=SEED)), ("clf", LR())])}
scoring = {"recall": "recall", "precision": "precision", "f1": "f1", "pr_auc": "average_precision"}
cv5 = StratifiedKFold(5, shuffle=True, random_state=SEED)
imb = pd.DataFrame({n: {k: v.mean() for k, v in cross_validate(p, X_train, y_train, cv=cv5, scoring=scoring).items() if k.startswith("test_")} for n, p in strategies.items()}).T
imb.columns = [c.replace("test_", "") for c in imb.columns]
display(imb.round(4))
imb[["recall", "precision", "f1", "pr_auc"]].plot(kind="bar", figsize=(9, 4)); plt.title("Imbalance strategies (CV on training set)"); plt.xticks(rotation=15); plt.ylim(0, 1.05); plt.show()
best_imb = imb["f1"].idxmax()
imb_note = ("class weights had the best cross-validated F1" if best_imb == "class_weight" else
            f"'{best_imb}' had the highest CV F1 at the 0.5 threshold, but F1 at 0.5 depends on the threshold, which we optimise later, so class weights were kept as the simple, leak-safe choice that favours recall")
interpret(f"Best CV F1: **{best_imb}** ({imb['f1'].max():.3f}). Best CV PR-AUC: **{imb['pr_auc'].idxmax()}** ({imb['pr_auc'].max():.3f}). Recall without handling is {imb.loc['No handling', 'recall']:.3f}; with the other strategies it ranges from {imb.drop('No handling')['recall'].min():.3f} to {imb.drop('No handling')['recall'].max():.3f}, usually at the cost of precision.",
          f"Decision: the main models use **class weights** (`scale_pos_weight` for boosting). Note: {imb_note}.")

---
# Phase 4: Machine Learning Track

We compare three models, all inside pipelines so preprocessing is learned from training data only:

1. **Logistic Regression**, the simple, explainable baseline for text.
2. **Random Forest**, many decision trees voting together.
3. **LightGBM**, gradient boosting (each new tree fixes the errors of the earlier ones), very strong on sparse text plus tabular features. We **tune** it with `RandomizedSearchCV`, optimising **PR-AUC** (not accuracy), and compare it before and after tuning.

Every model is scored on the same untouched test set at the default threshold of 0.5. Section 6 then looks at better thresholds.

In [ ]:
RESULTS = {}
def compute_metrics(y_true, prob, thr=0.5):
    pred = (np.asarray(prob) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {"Accuracy": accuracy_score(y_true, pred), "Precision": precision_score(y_true, pred, zero_division=0),
            "Recall": recall_score(y_true, pred, zero_division=0), "F1": f1_score(y_true, pred, zero_division=0),
            "ROC_AUC": roc_auc_score(y_true, prob), "PR_AUC": average_precision_score(y_true, prob), "TP": tp, "FP": fp, "FN": fn, "TN": tn}
def add_result(name, kind, prob, est=None):
    RESULTS[name] = {"kind": kind, "prob": np.asarray(prob).ravel(), "metrics": compute_metrics(y_test, prob), "estimator": est}
    m = RESULTS[name]["metrics"]
    print(f"{name:<22} Acc={m['Accuracy']:.4f} Prec={m['Precision']:.4f} Rec={m['Recall']:.4f} F1={m['F1']:.4f} ROC-AUC={m['ROC_AUC']:.4f} PR-AUC={m['PR_AUC']:.4f} | TP={m['TP']} FP={m['FP']} FN={m['FN']}")

SPW = n_neg / n_pos                        # real postings per fake posting in TRAIN, used by boosting
pipes = {"Logistic Regression": Pipeline([("prep", make_preprocessor()), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))]),
         "Random Forest": Pipeline([("prep", make_preprocessor()), ("clf", RandomForestClassifier(n_estimators=N_TREES, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED))]),
         "LightGBM": Pipeline([("prep", make_preprocessor()), ("clf", LGBMClassifier(n_estimators=N_TREES, learning_rate=0.05, class_weight="balanced", subsample=0.8, subsample_freq=1, colsample_bytree=0.6, n_jobs=-1, random_state=SEED, verbose=-1))])}
for name, p in pipes.items():
    p.fit(X_train, y_train); add_result(name, "ML", p.predict_proba(X_test)[:, 1], p)
b = pd.DataFrame({n: RESULTS[n]["metrics"] for n in RESULTS}).T
interpret(f"Accuracy only ranges from {b['Accuracy'].min():.3f} to {b['Accuracy'].max():.3f} (the 'always real' baseline gets {BASELINE_ACC:.3f}), while recall ranges from {b['Recall'].min():.3f} to {b['Recall'].max():.3f} and PR-AUC from {b['PR_AUC'].min():.3f} to {b['PR_AUC'].max():.3f}. Accuracy cannot separate good fraud models from poor ones.",
          f"Best baseline by PR-AUC: **{b['PR_AUC'].astype(float).idxmax()}**.")

In [ ]:
# Tune LightGBM with random search on the TRAINING set only; PR-AUC (average precision) decides, recall/precision/F1 are recorded too
space = {"clf__n_estimators": randint(150, 450), "clf__num_leaves": randint(15, 80), "clf__learning_rate": loguniform(0.02, 0.2),
         "clf__subsample": uniform(0.6, 0.4), "clf__colsample_bytree": uniform(0.4, 0.6), "clf__min_child_samples": randint(5, 40),
         "clf__reg_lambda": loguniform(1e-2, 10), "prep__text__max_features": [2000, 3000]}
search = RandomizedSearchCV(clone(pipes["LightGBM"]), space, n_iter=N_ITER, cv=StratifiedKFold(3, shuffle=True, random_state=SEED), random_state=SEED, n_jobs=1,
                            scoring={"pr_auc": "average_precision", "recall": "recall", "precision": "precision", "f1": "f1"}, refit="pr_auc")
t0 = time.time(); search.fit(X_train, y_train)
add_result("LightGBM (Tuned)", "ML", search.best_estimator_.predict_proba(X_test)[:, 1], search.best_estimator_)
bi = search.best_index_
before, after = RESULTS["LightGBM"]["metrics"], RESULTS["LightGBM (Tuned)"]["metrics"]
print("Best parameters:", {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in search.best_params_.items()}, f"| search time {(time.time() - t0) / 60:.1f} min")
d = after["PR_AUC"] - before["PR_AUC"]
interpret(f"Cross-validated results of the best setting (inside the training set): PR-AUC {search.cv_results_['mean_test_pr_auc'][bi]:.4f}, recall {search.cv_results_['mean_test_recall'][bi]:.4f}, precision {search.cv_results_['mean_test_precision'][bi]:.4f}.",
          f"Test PR-AUC moved from {before['PR_AUC']:.4f} to {after['PR_AUC']:.4f} ({d:+.4f}); recall from {before['Recall']:.3f} to {after['Recall']:.3f}. Tuning " + ("helped." if d > 0.005 else "did not clearly help." if d > -0.005 else "slightly hurt on the test set."),
          "Tuning does not always help. With only a few fake postings in the test set, small changes can be noise, and cross-validation numbers are the more stable evidence.")

### 4.1 ROC and Precision-Recall curves
The ROC curve is better when it hugs the top-left corner. The **Precision-Recall curve** is the stricter test for rare fraud: the dashed line shows what a random model would get (it equals the fake rate). For imbalanced problems the PR curve is the more honest of the two, because the huge number of real postings makes even a weak model look good on ROC.

In [ ]:
def plot_curves(names, title=""):
    fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
    for n in names:
        f, t, _ = roc_curve(y_test, RESULTS[n]["prob"]); ax[0].plot(f, t, label=f"{n} (AUC={RESULTS[n]['metrics']['ROC_AUC']:.3f})")
        p, r, _ = precision_recall_curve(y_test, RESULTS[n]["prob"]); ax[1].plot(r, p, label=f"{n} (AP={RESULTS[n]['metrics']['PR_AUC']:.3f})")
    ax[0].plot([0, 1], [0, 1], "k--"); ax[0].set(title="ROC curves" + title, xlabel="False positive rate", ylabel="Recall"); ax[0].legend(fontsize=8)
    ax[1].axhline(y_test.mean(), c="k", ls="--"); ax[1].set(title="Precision-Recall curves" + title, xlabel="Recall", ylabel="Precision"); ax[1].legend(fontsize=8)
    plt.tight_layout(); plt.show()
plot_curves(list(RESULTS))
interpret("A curve that stays above the others over the whole range is the better ranker; crossing curves mean the best model depends on the operating point.",
          f"The random-model line in the PR plot sits at {y_test.mean():.3f}. Curves far above it show real learning.")

---
# Phase 5: Deep Learning Track (Keras Neural Network)

Neural networks need dense, scaled input. Our preprocessing makes thousands of sparse text columns, which would give a network far more weights than the few hundred fake postings it can learn from. So we compress with **TruncatedSVD to 200 components** and apply `StandardScaler`. To avoid leakage, a **15% validation set** is cut from the training data first and the preprocessing is fitted on the remaining training part only. The test set stays untouched.

**Architecture (chosen for this data: a few thousand rows, few fake examples, 200 inputs, strong imbalance, so small and regularised).**

| Part | Choice | Reason |
|---|---|---|
| Input | 200 neurons | One per SVD component |
| Hidden layers | 128, 64, 32 neurons, ReLU | Learns feature combinations while narrowing step by step; ReLU trains fast |
| Batch normalisation | After layers 1 and 2 | Stabilises training |
| Dropout | 0.4, 0.3, 0.2 | Randomly switches neurons off so the network cannot memorise |
| Output | 1 neuron, sigmoid | Fraud probability between 0 and 1 |
| Loss / optimizer | Binary cross-entropy with class weights / Adam (lr 0.001) | Standard yes-no loss; weights handle imbalance |
| Batch size / epochs | 256 / up to 80 | Large batches are stable for rare classes |
| Callbacks | EarlyStopping on validation PR-AUC, ReduceLROnPlateau on validation loss | Stop when learning stops helping; lower the learning rate when progress stalls |

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.15, stratify=y_train, random_state=SEED)
dl_prep = Pipeline([("prep", make_preprocessor()), ("svd", TruncatedSVD(200, random_state=SEED)), ("scale", StandardScaler())])
X_tr_d = dl_prep.fit_transform(X_tr).astype("float32")                 # fitted on the training part only
X_val_d, X_te_d = dl_prep.transform(X_val).astype("float32"), dl_prep.transform(X_test).astype("float32")
cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_tr.values); class_weight = {0: float(cw[0]), 1: float(cw[1])}

keras.utils.set_random_seed(SEED)
ann = keras.Sequential([layers.Input(shape=(X_tr_d.shape[1],)),
                        layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4)), layers.BatchNormalization(), layers.Dropout(0.4),
                        layers.Dense(64, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4)), layers.BatchNormalization(), layers.Dropout(0.3),
                        layers.Dense(32, activation="relu"), layers.Dropout(0.2),
                        layers.Dense(1, activation="sigmoid")], name="fraud_ann")
ann.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
            metrics=["accuracy", keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall"),
                     keras.metrics.AUC(name="roc_auc"), keras.metrics.AUC(name="pr_auc", curve="PR")])
print("Class weights:", class_weight, "| fake postings in train/val/test:", int(y_tr.sum()), int(y_val.sum()), int(y_test.sum()))
ann.summary()

In [ ]:
cb = [callbacks.EarlyStopping(monitor="val_pr_auc", mode="max", patience=10, restore_best_weights=True, verbose=1),
      callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5, verbose=1)]
history = ann.fit(X_tr_d, y_tr.values, validation_data=(X_val_d, y_val.values), epochs=EPOCHS, batch_size=256,
                  class_weight=class_weight, callbacks=cb, verbose=2)
hist = pd.DataFrame(history.history); print(f"Stopped after {len(hist)} epochs")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for a, (m, t) in zip(ax, [("loss", "Loss"), ("pr_auc", "PR-AUC"), ("recall", "Recall (threshold 0.5)")]):
    a.plot(hist[m], label="Training", c=BLUE); a.plot(hist[f"val_{m}"], label="Validation", c=RED); a.set(title=t, xlabel="Epoch"); a.legend()
plt.tight_layout(); plt.show()

dl_prob = ann.predict(X_te_d, batch_size=1024, verbose=0).ravel(); val_prob = ann.predict(X_val_d, batch_size=1024, verbose=0).ravel()
DL_NAME = "Deep Learning (ANN)"; add_result(DL_NAME, "DL", dl_prob)
print(classification_report(y_test, (dl_prob >= 0.5).astype(int), target_names=["Real", "Fake"], digits=4, zero_division=0))
be = int(hist["val_pr_auc"].idxmax())
interpret(f"Best epoch by validation PR-AUC: {be + 1} (validation {hist['val_pr_auc'].max():.3f}, training {hist['pr_auc'].iloc[be]:.3f}, gap {hist['pr_auc'].iloc[be] - hist['val_pr_auc'].iloc[be]:+.3f}). At the last epoch, validation loss minus training loss is {hist['val_loss'].iloc[-1] - hist['loss'].iloc[-1]:+.3f}.",
          "**Overfitting check:** overfitting shows as training metrics that keep improving while validation metrics stall or worsen. Early stopping with restored best weights, dropout and L2 limit it; the gaps above show how much is left. Training metrics also look better partly because dropout is active only in training.")

---
# Phase 6: Evaluation, Model Selection and Threshold

**Reading the confusion matrix here.** TP = fake posting caught. TN = real posting left alone. FP = real posting wrongly flagged (a genuine employer is delayed and a reviewer loses time). FN = fake posting missed (a scam stays online and job seekers can be harmed). **False negatives are usually the costlier error**, so we pay close attention to recall while keeping precision at a level the review team can handle.

We now put all models side by side. The **final model** is the one with the highest **PR-AUC** (the best threshold-free measure for rare fraud). We also show which model is cheapest under the assumed costs, `COST_FN` and `COST_FP`, which are illustrative, not real company numbers.

In [ ]:
comparison = pd.DataFrame({n: r["metrics"] for n, r in RESULTS.items()}).T
comparison = comparison.astype(float).assign(Type=[r["kind"] for r in RESULTS.values()])[["Type"] + list(next(iter(RESULTS.values()))["metrics"])]
comparison["Expected_cost"] = comparison["FN"] * COST_FN + comparison["FP"] * COST_FP
comparison = comparison.sort_values("PR_AUC", ascending=False)
display(comparison.style.format({c: "{:.4f}" for c in ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"]}).background_gradient(subset=["Recall", "F1", "PR_AUC"], cmap="Greens"))

names = list(comparison.index)
fig, axes = plt.subplots(1, len(names), figsize=(4.1 * len(names), 3.8))
for a, n in zip(np.atleast_1d(axes), names):
    m = RESULTS[n]["metrics"]; sns.heatmap([[m["TN"], m["FP"]], [m["FN"], m["TP"]]], annot=True, fmt=".0f", cmap="Blues", cbar=False, ax=a,
                                           xticklabels=["Pred real", "Pred fake"], yticklabels=["Actual real", "Actual fake"]); a.set_title(f"{n}\nRecall={m['Recall']:.2f}", fontsize=9)
plt.tight_layout(); plt.show()
plot_curves([DL_NAME] + [n for n in names if RESULTS[n]["kind"] == "ML"][:2], " (ANN vs top ML models)")

best_ml_name = comparison[comparison["Type"] == "ML"]["PR_AUC"].idxmax()
FINAL_NAME = comparison["PR_AUC"].idxmax()
best_business = comparison.sort_values(["Expected_cost", "PR_AUC"], ascending=[True, False]).index[0]
fm, bm, dm = comparison.loc[FINAL_NAME], comparison.loc[best_ml_name], comparison.loc[DL_NAME]
interpret(f"**Final model: {FINAL_NAME}** (highest PR-AUC {fm['PR_AUC']:.4f}); at 0.5 it has recall {fm['Recall']:.3f}, precision {fm['Precision']:.3f}, F1 {fm['F1']:.3f}.",
          f"Best recall: {comparison['Recall'].idxmax()} ({comparison['Recall'].max():.3f}). Best precision: {comparison['Precision'].idxmax()} ({comparison['Precision'].max():.3f}). Best F1: {comparison['F1'].idxmax()} ({comparison['F1'].max():.3f}). Cheapest under the assumed costs at 0.5: **{best_business}**.",
          f"Best ML model: {best_ml_name} (PR-AUC {bm['PR_AUC']:.4f}). Neural network: PR-AUC {dm['PR_AUC']:.4f}, which " + ("beat" if dm["PR_AUC"] > bm["PR_AUC"] else "did **not** beat") + " the best ML model. On a few thousand rows of text plus tabular data this is a normal, honest result.",
          "Close scores can swap places on another split because the test set has few fake postings. Do not over-read tiny gaps.")

### 6.1 Threshold optimisation
A model outputs a probability, and the threshold decides when it becomes a "fake" flag. The default 0.5 is arbitrary. A **lower threshold** catches more fraud (higher recall) but creates more false alarms; a **higher threshold** gives fewer false alarms but lets more scams through. We choose the threshold on data the final model did **not** score for the final test result (5-fold out-of-fold predictions on the training set for an ML model, the validation split for the neural network), then apply it once to the test set. Choosing it on the test set would make the result look better than it is.

In [ ]:
if RESULTS[FINAL_NAME]["kind"] == "ML":
    sel_prob = cross_val_predict(clone(RESULTS[FINAL_NAME]["estimator"]), X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
    sel_y, sel_label = y_train.values, "out-of-fold predictions on the training set"
else:
    sel_prob, sel_y, sel_label = val_prob, y_val.values, "validation split"

def sweep(yt, p):
    rows = []
    for t in np.round(np.arange(0.02, 0.99, 0.01), 2):
        m = compute_metrics(yt, p, t) if len(set(yt)) > 1 else None
        pr = m["TP"] / (m["TP"] + m["FP"]) if (m["TP"] + m["FP"]) else 0.0
        rows.append({"threshold": t, "precision": pr, "recall": m["Recall"], "f1": m["F1"], "cost": m["FN"] * COST_FN + m["FP"] * COST_FP})
    return pd.DataFrame(rows)
sw = sweep(sel_y, sel_prob)
thr_f1 = float(sw.loc[sw["f1"].idxmax(), "threshold"])
thr_cost = float(np.median(sw.loc[sw["cost"] == sw["cost"].min(), "threshold"]))
ok = sw[sw["recall"] >= 0.90]; thr_r90 = float(ok["threshold"].max()) if len(ok) else float(sw.loc[sw["recall"].idxmax(), "threshold"])
RECOMMENDED_THR = thr_cost

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(sw["threshold"], sw["precision"], label="Precision", c=BLUE); ax[0].plot(sw["threshold"], sw["recall"], label="Recall", c=RED); ax[0].plot(sw["threshold"], sw["f1"], "g--", label="F1")
for t, l in [(thr_f1, "max F1"), (thr_cost, "min cost")]: ax[0].axvline(t, c="grey", ls=":"); ax[0].text(t, 0.02, l, rotation=90, fontsize=8)
ax[0].set(title=f"Precision, recall, F1 vs threshold\n({sel_label})", xlabel="Threshold"); ax[0].legend()
ax[1].plot(sw["threshold"], sw["cost"] / len(sel_y) * 1000, c="black"); ax[1].set(title=f"Assumed cost per 1000 postings (FN={COST_FN}, FP={COST_FP})", xlabel="Threshold")
plt.tight_layout(); plt.show()

final_prob = RESULTS[FINAL_NAME]["prob"]
tt = pd.DataFrame({lbl: {**compute_metrics(y_test, final_prob, t), "Threshold": t} for lbl, t in {"Default 0.5": 0.5, "Max F1": thr_f1, "Lowest assumed cost": thr_cost, "Recall >= 0.90": thr_r90}.items()}).T
tt["Flagged per 1000"] = (tt["TP"] + tt["FP"]) / len(y_test) * 1000
display(tt[["Threshold", "Precision", "Recall", "F1", "TP", "FP", "FN", "Flagged per 1000"]].astype(float).round(4))
rc, d5 = tt.loc["Lowest assumed cost"], tt.loc["Default 0.5"]
interpret(f"Thresholds chosen on the selection set: max F1 {thr_f1:.2f}, lowest assumed cost {thr_cost:.2f}, recall of at least 0.90 at {thr_r90:.2f}. **Recommended: {RECOMMENDED_THR:.2f}** (lowest assumed cost).",
          f"On the test set it gives recall {rc['Recall']:.3f} and precision {rc['Precision']:.3f}, against recall {d5['Recall']:.3f} and precision {d5['Precision']:.3f} at 0.5, flagging about {rc['Flagged per 1000']:.1f} postings per 1000.",
          "**Maximum fraud catch vs fewer false alarms:** a low threshold maximises catches but floods reviewers; a high one keeps the queue short but lets scams through. The right point depends on the real cost ratio, which is why the costs at the top are editable.")

---
# Phase 7: Explainability and Business Interpretation

A fraud team must be able to explain **why** a posting was flagged. For the best ML model we use three views: **built-in feature importance** on the expanded features, **permutation importance on the original columns** (shuffle one column in the test set and measure how much PR-AUC drops, which works for any model and gives business-level importance), and **SHAP** (shows direction and size of each feature's effect; skipped safely if it fails in your environment).

In [ ]:
est = RESULTS[best_ml_name]["estimator"]; prep, clf = est.named_steps["prep"], est.named_steps["clf"]
feat_names = prep.get_feature_names_out()
perm_cols = [TEXT_FEATURE] + NUMERIC_FEATURES + CATEGORICAL_FEATURES + BINARY_FEATURES
perm = permutation_importance(est, X_test[perm_cols], y_test, scoring="average_precision", n_repeats=5, random_state=SEED, n_jobs=1)
perm_df = pd.Series(perm.importances_mean, index=perm_cols).sort_values(ascending=False)
native = pd.Series(clf.feature_importances_ if hasattr(clf, "feature_importances_") else np.abs(clf.coef_.ravel()), index=feat_names).sort_values(ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
perm_df.head(12)[::-1].plot(kind="barh", color=RED, ax=ax[0]); ax[0].set_title(f"{best_ml_name}: permutation importance (drop in PR-AUC)")
native.head(15)[::-1].plot(kind="barh", color=BLUE, ax=ax[1]); ax[1].set_title("Built-in importance (expanded features)")
plt.tight_layout(); plt.show()
top_perm_names, native_top = perm_df.head(5).index.tolist(), native.head(6).index.tolist()
interpret(f"Most important original columns: **{', '.join(top_perm_names)}**. Shuffling `{perm_df.index[0]}` lowers PR-AUC by {perm_df.iloc[0]:.4f}, more than any other column.",
          f"Top expanded features: {', '.join(native_top)} (`text__` = a word or word pair, `num__` = numeric, `cat__` = category, `bin__` = flag).",
          "Values near zero mean the model barely uses that column. Importance shows association, not cause, and scammers can change their wording over time.")

In [ ]:
# SHAP: global summary plus one local explanation (the highest-scoring truly fake test posting)
try:
    import shap
    fake_idx = np.where(y_test.values == 1)[0]; top_local = fake_idx[np.argmax(RESULTS[best_ml_name]["prob"][fake_idx])]
    rows = sorted(set(np.random.RandomState(SEED).choice(len(X_test), min(300, len(X_test)), replace=False).tolist() + [int(top_local)]))
    Xs = prep.transform(X_test.iloc[rows]); Xs = Xs.toarray() if hasattr(Xs, "toarray") else np.asarray(Xs)
    expl = shap.TreeExplainer(clf); sv = expl.shap_values(Xs)
    sv = sv[1] if isinstance(sv, list) else np.asarray(sv); sv = sv[:, :, 1] if sv.ndim == 3 else sv
    shap.summary_plot(sv, Xs, feature_names=feat_names, max_display=15, show=False); plt.title("SHAP summary"); plt.tight_layout(); plt.show()
    i = rows.index(int(top_local)); ev = np.ravel(expl.expected_value)[-1]
    shap.plots.waterfall(shap.Explanation(values=sv[i], base_values=ev, data=Xs[i], feature_names=list(feat_names)), max_display=10, show=False); plt.tight_layout(); plt.show()
    interpret("In the summary, each dot is one posting; red = high feature value, blue = low; dots on the right push the prediction towards **fake**.",
              f"The waterfall explains one confirmed fake posting (title: '{df_fe.loc[X_test.index[top_local], 'title'][:60]}'): bars to the right raised its fraud score, bars to the left lowered it. A reviewer can read this as the reason for the flag.")
except Exception as err:
    print("SHAP was skipped:", repr(err), "\nThe permutation importance above is already a valid explanation.")

### 7.1 What this means for the business
**What the model predicts:** a fraud probability between 0 and 1 for every new job posting. **How a job portal would use it:** postings below the threshold go live normally; postings above it go to a review queue **sorted from the highest score down**, so reviewers see the riskiest first; confirmed scams are removed and applicants can be warned; reviewer decisions are saved and used to retrain the model. **False positives** delay a genuine employer and cost reviewer time. **False negatives** leave a scam online, which can cost job seekers money or personal data and damages trust, so they are usually the more expensive mistake. The model does **not replace** reviewers: it focuses their limited time and the explanations above tell them why a posting looks risky.

In [ ]:
mt = compute_metrics(y_test, final_prob, RECOMMENDED_THR); n_test = len(y_test)
band = pd.cut(final_prob, [-0.01, RECOMMENDED_THR, max(0.80, RECOMMENDED_THR + 0.01), 1.01], labels=["Low (publish)", "Medium (review)", "High (urgent review)"])
bt = pd.DataFrame({"band": band, "actual": y_test.values}).groupby("band", observed=False)["actual"].agg(postings="size", actual_fake="sum")
bt["fake_rate_%"] = (bt["actual_fake"] / bt["postings"].replace(0, np.nan) * 100).round(2); display(bt)
view = df_fe.loc[X_test.index, ["title", "country"]].assign(fraud_probability=final_prob, actual=y_test.map({0: "Real", 1: "Fake"}).values)
print("TOP 10 HIGHEST-RISK TEST POSTINGS"); display(view.sort_values("fraud_probability", ascending=False).head(10))
interpret(f"At threshold {RECOMMENDED_THR:.2f} the model sends **{(mt['TP'] + mt['FP']) / n_test * 1000:.1f} of every 1000 postings** to review. It catches **{mt['TP']} of {mt['TP'] + mt['FN']}** fake postings in the test set, misses **{mt['FN']}** and raises **{mt['FP']}** false alarms.",
          f"A reviewer checks about {(mt['TP'] + mt['FP']) / max(mt['TP'], 1):.1f} flagged postings to find one real scam. Probabilities are not exact percentages because class weights were used; use them for **ranking** (the 0.80 band cut-off is an example you can change).")

---
# Conclusion

In [ ]:
f05 = compute_metrics(y_test, final_prob, 0.5); fth = compute_metrics(y_test, final_prob, RECOMMENDED_THR)
dl_m, ml_m = RESULTS[DL_NAME]["metrics"], RESULTS[best_ml_name]["metrics"]
display(Markdown(f"""
**Problem.** Detect fake job postings, a rare but harmful kind of fraud, so a portal can review the riskiest postings first.

**Dataset.** The file had {len(raw_df):,} rows; after cleaning (ID dropped, {len(raw_df) - len(df_model):,} duplicate or unusable rows removed) we modelled {len(df_model):,} postings, **{n_pos_all:,} ({fraud_rate * 100:.2f}%) fake**, about {imbalance_ratio:.1f} real per fake. Always predicting 'real' would be {BASELINE_ACC * 100:.2f}% accurate and catch nothing.

**EDA findings.** Largest missing-field gap: {gap.iloc[0]['column']} (fraud {gap.iloc[0]['fraud_if_missing']:.2f}% if missing vs {gap.iloc[0]['fraud_if_present']:.2f}% if present). Strongest linear correlation with fraud: {tcorr.index[0]} ({tcorr.iloc[0]:.3f}). Words typical of fake postings: {', '.join(wt['word'].tail(5)[::-1])}.

**Important features.** Permutation importance: {', '.join(top_perm_names)}. Built-in importance: {', '.join(native_top[:4])}.

**Models and imbalance.** Logistic Regression, Random Forest, LightGBM (baseline and tuned) and a Keras neural network. Imbalance handled mainly with class weights; in section 3, {imb_note}.

**Best ML model.** {best_ml_name}: PR-AUC {ml_m['PR_AUC']:.4f}, recall {ml_m['Recall']:.4f}, precision {ml_m['Precision']:.4f}, F1 {ml_m['F1']:.4f} at 0.5. **Deep learning:** PR-AUC {dl_m['PR_AUC']:.4f}, recall {dl_m['Recall']:.4f}, precision {dl_m['Precision']:.4f}, which was {"better" if dl_m['PR_AUC'] > ml_m['PR_AUC'] else "not better"} than the best ML model.

**Final model: {FINAL_NAME}** (highest PR-AUC). At 0.5: recall {f05['Recall']:.4f}, precision {f05['Precision']:.4f}, F1 {f05['F1']:.4f}, ROC-AUC {f05['ROC_AUC']:.4f}, PR-AUC {f05['PR_AUC']:.4f}. At the recommended threshold **{RECOMMENDED_THR:.2f}**: recall {fth['Recall']:.4f}, precision {fth['Precision']:.4f}, catching {fth['TP']} of {fth['TP'] + fth['FN']} fake postings with {fth['FP']} false alarms.

**Business impact.** Reviewers check about {(fth['TP'] + fth['FP']) / len(y_test) * 1000:.1f} postings per 1000 instead of reading everything, while {fth['FN']} of {fth['TP'] + fth['FN']} fake postings in the test set are missed.
"""))

### Limitations
- **Small number of fake postings:** metrics such as recall move a lot with one or two postings, so all results carry uncertainty.
- **Representativeness and drift:** the data is one collection of mostly English postings from a limited period. Scammers change their wording, so the model needs monitoring and retraining.
- **Leakage risks:** exact duplicates were removed and every learned step sits in a pipeline, but slightly edited copies of one posting can survive and make the test score a little optimistic. There is no date column, so a time-based split was not possible.
- **Label quality and fairness:** labels depend on how the dataset creators judged "fake", and features like `country` can bias against genuine employers from some places.
- **Errors and deployment:** false positives annoy real employers, false negatives leave scams online, scoring must match the training pipeline exactly in real time, and SVD inputs (used for the neural network) are harder to explain.
- **Assumed costs:** the business costs used for the threshold are illustrative.

### Future scope
Language-model text embeddings instead of TF-IDF, near-duplicate detection and group-based splits, cost-sensitive learning, probability calibration, anomaly detection (Isolation Forest or an autoencoder) for new scam styles, more boosting models (XGBoost, CatBoost) and ensembling, employer-level behaviour features if account ids exist, a real-time API with a review feedback loop, and monitoring with drift detection.

---
# Interview / Viva Preparation
The answers below are built from **your own run**, so the numbers match your notebook. Read them once after running everything, then practise saying them in your own words.

In [ ]:
gain = comparison.loc["LightGBM (Tuned)", "PR_AUC"] - comparison.loc["LightGBM", "PR_AUC"]
vs = "did not beat" if dl_m["PR_AUC"] <= ml_m["PR_AUC"] else "beat"
qa = [
 ("1. Explain your fraud detection project.", f"I built a system that scores job postings for fraud risk. Using the Kaggle fake job posting data ({len(df_model):,} postings after cleaning, {fraud_rate * 100:.2f}% fake), I cleaned and explored the data, engineered features from the text and company details, handled the imbalance, compared Logistic Regression, Random Forest, tuned LightGBM and a neural network, chose a threshold using business costs, and explained the predictions."),
 ("2. Why fraud detection?", "Fraud is rare but costly, so it teaches imbalance, the right metrics, leakage and business cost. Fake postings also harm real job seekers."),
 ("3. What is the target variable?", f"`fraudulent` (1 = fake, 0 = real): {n_pos_all:,} fake and {n_neg_all:,} real postings."),
 ("4. What preprocessing did you do?", f"Dropped the ID, turned blank text into missing values, removed duplicate rows, added missing-value flags, and used a pipeline: TF-IDF for text, RobustScaler for numbers, one-hot for categories, flags passed through, all fitted on training data only."),
 ("5. Why did you use SMOTE?", f"I tested it as one of four strategies, inside a pipeline so only training folds were resampled. For the main models I used class weights: {imb_note}."),
 ("6. Why is accuracy not enough?", f"Because fraud is rare: always predicting 'real' gets {BASELINE_ACC * 100:.2f}% accuracy on my test set and catches no fraud. So I use precision, recall, F1, ROC-AUC and PR-AUC."),
 ("7. Explain precision and recall.", f"Precision is the share of flagged postings that are really fake; recall is the share of all fake postings we catch. At the recommended threshold my final model has precision {fth['Precision']:.3f} and recall {fth['Recall']:.3f}."),
 ("8. Why is recall important?", f"A missed scam stays online and can hurt a job seeker, while a false alarm costs a few minutes of review. The final model missed {fth['FN']} of {fth['TP'] + fth['FN']} fake postings at the recommended threshold."),
 ("9. What is ROC-AUC?", f"How well the model ranks a fake posting above a real one across all thresholds (1.0 perfect, 0.5 random). Mine is {f05['ROC_AUC']:.3f}; I also use PR-AUC ({f05['PR_AUC']:.3f}), which is stricter for rare classes."),
 ("10. Why did you select your final model?", f"{FINAL_NAME} had the highest PR-AUC ({f05['PR_AUC']:.3f}), the best threshold-free measure for imbalanced data. I then tuned the threshold for business cost separately."),
 ("11. Why compare ML and DL?", f"To check whether a neural network adds value. It {vs} the best ML model ({best_ml_name}): PR-AUC {dl_m['PR_AUC']:.3f} vs {ml_m['PR_AUC']:.3f}. On a few thousand rows of text plus tabular data, boosting is often as good and easier to explain."),
 ("12. What is overfitting and how did you prevent it?", f"Overfitting is memorising training data and doing worse on new data. I used a held-out test set, cross-validation for tuning, dropout, batch normalisation, L2 and early stopping (best epoch {be + 1}), and removed duplicates."),
 ("13. What is data leakage?", "Information from outside the training data reaching the model so scores look better than reality. I avoided it by removing duplicates before splitting, fitting every transformer inside pipelines, resampling only training folds, and choosing the threshold on out-of-fold or validation predictions, not on the test set."),
 ("14. How would you deploy it?", "Save the whole pipeline and the threshold, serve it as an API that scores each new posting, send high scores to a review queue, log reviewer decisions, monitor score distribution and precision, and retrain regularly."),
 ("15. What if fraud patterns change?", "That is concept drift: accuracy slowly drops. I would monitor the flagged rate and reviewed precision, retrain on newly reviewed postings, and add anomaly detection for new scam styles."),
 ("16. False positives and false negatives?", f"A false positive is a real posting wrongly flagged ({fth['FP']} in my test set at the recommended threshold); a false negative is a fake posting missed ({fth['FN']})."),
 ("17. What are the most important features?", f"By permutation importance on {best_ml_name}: {', '.join(top_perm_names[:4])}. They show what a reviewer should check first."),
 ("18. How would a company use your model?", f"Score each new posting; at threshold {RECOMMENDED_THR:.2f}, about {(fth['TP'] + fth['FP']) / len(y_test) * 1000:.1f} per 1000 go to review, highest risk first. Reviewers confirm and remove scams, and their decisions feed retraining."),
 ("19. How would you improve it?", "Language-model embeddings, near-duplicate detection, cost-sensitive learning, calibration, anomaly detection, monitoring and SHAP explanations for reviewers."),
 ("20. Did tuning help?", f"Test PR-AUC for LightGBM changed by {gain:+.4f} after tuning. With few fake postings in the test set, small changes are within noise."),
]
for q, a in qa: display(Markdown(f"**{q}**\n\n{a}\n"))